In [ ]:
import re
import html
import torch
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


# ===== PATH CONFIG =====
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_PATH = "./weights/lora_svg_textsvg_final"
TEST_PATH = "test.csv"
OUTPUT_PATH = "submission.csv"


# ===== LOAD MODEL =====
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

test_df = pd.read_csv(TEST_PATH)

print("model loaded")
print("test size:", len(test_df))


# ===== SVG UTILS =====
SVG_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
SVG_FOOTER = '</svg>'


def fallback_svg(prompt):
    p = str(prompt).lower()

    if any(w in p for w in ["arrow", "icon", "symbol"]):
        return SVG_HEADER + '''
<rect x="40" y="40" width="176" height="176" fill="none" stroke="black" stroke-width="8"/>
<path d="M80 128 Q128 60 176 128" fill="none" stroke="black" stroke-width="8"/>
<polygon points="160,112 180,128 160,144" fill="black"/>
''' + SVG_FOOTER

    if any(w in p for w in ["person", "sitting", "chair"]):
        return SVG_HEADER + '''
<circle cx="128" cy="80" r="20" fill="black"/>
<line x1="128" y1="100" x2="128" y2="160" stroke="black" stroke-width="8"/>
<line x1="128" y1="120" x2="90" y2="140" stroke="black" stroke-width="8"/>
<line x1="128" y1="120" x2="166" y2="140" stroke="black" stroke-width="8"/>
<rect x="90" y="150" width="76" height="10" fill="black"/>
''' + SVG_FOOTER

    if any(w in p for w in ["line", "lines", "horizontal", "vertical", "stripe"]):
        return SVG_HEADER + '''
<line x1="40" y1="60" x2="216" y2="60" stroke="black" stroke-width="6"/>
<line x1="60" y1="100" x2="196" y2="100" stroke="black" stroke-width="8"/>
<line x1="50" y1="140" x2="206" y2="140" stroke="black" stroke-width="4"/>
<line x1="70" y1="180" x2="186" y2="180" stroke="black" stroke-width="10"/>
''' + SVG_FOOTER

    if "triangle" in p:
        return SVG_HEADER + '''
<polygon points="128,60 60,196 196,196" fill="black"/>
''' + SVG_FOOTER

    if any(w in p for w in ["square", "rectangle"]):
        return SVG_HEADER + '''
<rect x="64" y="64" width="128" height="128" fill="black"/>
''' + SVG_FOOTER

    if any(w in p for w in ["circle", "circular"]):
        return SVG_HEADER + '''
<circle cx="128" cy="128" r="64" fill="black"/>
''' + SVG_FOOTER

    if any(w in p for w in ["geometric", "abstract"]):
        return SVG_HEADER + '''
<rect x="60" y="60" width="40" height="40" fill="black"/>
<circle cx="160" cy="100" r="24" fill="black"/>
<polygon points="128,180 100,220 156,220" fill="black"/>
''' + SVG_FOOTER

    if any(w in p for w in ["wood", "log", "firewood"]):
        return SVG_HEADER + '''
<rect x="60" y="120" width="136" height="24" fill="black"/>
<rect x="70" y="96" width="120" height="24" fill="black"/>
<circle cx="70" cy="132" r="12" fill="white"/>
<circle cx="190" cy="132" r="12" fill="white"/>
''' + SVG_FOOTER

    return SVG_HEADER + '''
<rect x="64" y="64" width="128" height="128" fill="black"/>
''' + SVG_FOOTER


def build_prompt(prompt):
    return f"Text: {prompt}\nSVG:"


def extract_svg(text):
    text = html.unescape(str(text))
    text = re.sub(r"```(?:svg|xml)?", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "")

    start = text.find("<svg")
    if start != -1:
        text = text[start:]

    m = re.search(r"<svg[\s\S]*?</svg>", text, flags=re.IGNORECASE)
    if m:
        return m.group(0)

    return None


def clean_svg(svg):
    svg = str(svg)
    svg = re.sub(r"[\x00-\x1F]", "", svg)
    svg = re.sub(r"\s+", " ", svg).strip()
    return svg


def normalize_svg(svg):
    return re.sub(r'<svg[^>]*>', SVG_HEADER, svg, count=1, flags=re.IGNORECASE)


def ensure_closed(svg):
    if "</svg>" not in svg:
        svg += SVG_FOOTER
    return svg


def is_valid_svg(svg):
    try:
        ET.fromstring(svg)
        return True
    except:
        return False


@torch.no_grad()
def generate_svg(prompt):
    inputs = tokenizer(
        build_prompt(prompt),
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.65,
        top_p=0.9
    )

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    svg = extract_svg(text)

    if svg is None:
        return fallback_svg(prompt)

    svg = clean_svg(svg)
    svg = normalize_svg(svg)
    svg = ensure_closed(svg)

    if not is_valid_svg(svg):
        return fallback_svg(prompt)

    return svg


# ===== GENERATE SUBMISSION =====
prompts = test_df["prompt"].tolist()
ids = test_df["id"].tolist()

preds = []
for i in tqdm(range(len(prompts))):
    preds.append(generate_svg(prompts[i]))

submission = pd.DataFrame({
    "id": ids,
    "svg": preds
})

submission.to_csv(OUTPUT_PATH, index=False)

print("saved to", OUTPUT_PATH)